In [ ]:
import dvuploader as dv
from pathlib import Path
from dotenv import load_dotenv
from itertools import batched
import os

load_dotenv()

dataverse_token = os.getenv("DATAVERSE_TOKEN")
server_url = "https://data.goettingen-research-online.de/"
dataset_doi = "https://doi.org/10.25625/GDCXQK"

scratch_folder = Path(os.getenv("SCRATCH_FOLDER")) / "data_export"

In [ ]:
finished_tables = [
    "chats",
    "chat_language",
    "chats_users",
    "entity_hashtags",
    "entity_urls",
    "poll_options",
    "polls",
    "reactions",
    "users",
    "messages",
]

dv.config(
    max_retries=10,
    max_retry_time=300,  # max wait time in seconds
    min_retry_time=2,  # initial wait time in seconds
    retry_multiplier=0.2,  # backoff multiplier
)

files_to_upload = []
for table in finished_tables:
    if (scratch_folder / f"{table}.parquet").is_file():
        print(f"Adding file {table}.parquet to DV uploader")
        filepath = str(scratch_folder / f"{table}.parquet")
        files_to_upload.append(dv.File(filepath=filepath))

    elif (scratch_folder / table).is_dir():
        folder = scratch_folder / table
        print(f"Adding folder {folder} to DV uploader")
        files = dv.add_directory(folder, rootDirectoryLabel=table)
        files_to_upload.extend(files)


# upload files in batches
for file_batch in batched(files_to_upload, 10):
    print(f"Uploading files {file_batch}...")
    dvuploader = dv.DVUploader(files=file_batch)
    dvuploader.upload(
        api_token=dataverse_token,
        dataverse_url=server_url,
        persistent_id=dataset_doi,
    )
    print("Done")

In [ ]:
# file = scratch_folder / "messages" / "messages_batch_2.parquet"
# files = [dv.File(filepath=str(file), directory_label="messages")]

# dvuploader = dv.DVUploader(files=files)
# dvuploader.upload(
#     api_token=dataverse_token,
#     dataverse_url="https://data.goettingen-research-online.de/",
#     persistent_id="https://doi.org/10.25625/GDCXQK",
# )

In [ ]:
import requests

headers = {"X-Dataverse-key": dataverse_token}


# Get all files in the dataset
response = requests.get(
    f"{server_url}/api/datasets/:persistentId/versions/:draft/files",
    params={"persistentId": dataset_doi},
    headers=headers,
)

files = response.json()["data"]

# Delete files matching the target directory
for file in files:
    if file.get("directoryLabel") == "messages":
        file_id = file["dataFile"]["id"]
        requests.delete(f"{server_url}/api/files/{file_id}", headers=headers)